# CIM XML to .cimtbl

`to_documents` turns any cim-graph model into `.cimtbl`, and the result loads back into an equal model. This notebook reads a CIM RDF/XML file, writes it as `.cimtbl`, loads that, and compares the two.

## A CIM XML file

Any CIM XML in a profile cim-graph knows will do. To keep this notebook self-contained, it makes one from the quick start's feeder.

In [1]:
import tempfile
from pathlib import Path

import cimtables
from cimgraph.utils import write_xml

work = Path(tempfile.mkdtemp())
write_xml(cimtables.load_model('tiny_feeder.cimtbl'), str(work / 'feeder.xml'))

Read it with cim-graph:

In [2]:
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel

from_xml = FeederModel(connection=XMLFile(filename=str(work / 'feeder.xml')), container=None, distributed=False)
sum(len(objects) for objects in from_xml.graph.values())

30

## Write it as .cimtbl

`to_documents` returns the documents; `format_document` shows one as text. `units` picks the units to write a column in (otherwise a readable one is chosen).

In [3]:
from cimtables.document import format_document

documents = cimtables.to_documents(from_xml, 'feeder', units={'length': 'ft', 'p': 'kW', 'q': 'kVAr'})
print(format_document(documents.master))

Profile = cim18v17
Authority = https://example.com/models
Model = tiny_feeder

Object BasePower: mRID=sbase, basePower (MVA)=100

Object BaseVoltage: mRID=bv_12kv, nominalVoltage (kV)=12.47

Object Substation: mRID=sub

Object Feeder: mRID=fdr, NormalEnergizingSubstation=sub

Table ConnectivityNode: mRID, ConnectivityNodeContainer
    n1,        fdr
    n2,        fdr
    sourcebus, fdr

Table ACLineSegment: mRID, node1, node2, phases, EquipmentContainer, BaseVoltage, length (ft), r, x
    ln_branch, n1,        n2, C,   fdr, bv_12kv, 500,  0.12, 0.2
    ln_trunk,  sourcebus, n1, ABC, fdr, bv_12kv, 2000, 0.38, 0.75

Table EnergyConsumer: mRID, node, phases, EquipmentContainer, BaseVoltage, p (kW), q (kVAr)
    ld_n1, n1, ABC, fdr, bv_12kv, 1155, 660
    ld_n2, n2, C,   fdr, bv_12kv, 170,  80

Object EnergySource: mRID=source, node=sourcebus, phases=ABC, EquipmentContainer=fdr, BaseVoltage=bv_12kv, nominalVoltage (kV)=12.47



The Terminals and per-phase objects were folded back into `node1`, `node2` and `phases` columns. Pass `terminals='explicit'` to keep every Terminal and per-phase object as its own row instead:

In [4]:
explicit = cimtables.to_documents(from_xml, 'feeder', terminals='explicit')
text = format_document(explicit.master)
print(text[text.index('Table Terminal'):])

Table Terminal: mRID, phases, sequenceNumber, ConductingEquipment, ConnectivityNode
    ld_n1_T1,     ABC, 1, ld_n1,     n1
    ld_n2_T1,     C,   1, ld_n2,     n2
    ln_branch_T1, C,   1, ln_branch, n1
    ln_branch_T2, C,   2, ln_branch, n2
    ln_trunk_T1,  ABC, 1, ln_trunk,  sourcebus
    ln_trunk_T2,  ABC, 2, ln_trunk,  n1
    source_T1,    ABC, 1, source,    sourcebus



## Load it back and compare

`write_documents` writes the files; `diff_models` compares two models object by object, by class and mRID. An empty list means they are equal.

In [5]:
from cimtables.document import write_documents

master = write_documents(documents, work / 'cimtbl')
from_cimtbl = cimtables.load_model(master)
cimtables.diff_models(from_xml, from_cimtbl)

[]

A change shows up as a line per difference:

In [6]:
from_cimtbl.find_by_mRID('ld_n2').p = from_cimtbl.cim.ActivePower(175, 'kW')
cimtables.diff_models(from_xml, from_cimtbl)

['EnergyConsumer ld_n2.p: expected 170000.0 watt, got 175000.0 watt']